Notebook ini memiliki beberapa fungsi:
1. Mount Google Drive & baca semua CSV review dari 3 platform
2. Konversi tanggal relatif Traveloka → tanggal absolut
3. Hapus review kosong
4. **Deteksi bahasa** (pakai `lingua-language-detector`)

5. **Translate semua review non-Indonesia → Indonesia** (en, ja, ko, ar, dll → id)
6. Export `dataset_absa_santika.csv`

**Kolom output:** `review_id, platform, hotel_name, text_review, text_review_original, date, original_language`

> `text_review` = teks final (sudah ditranslate ke Indonesia jika perlu)
> `text_review_original` = teks asli sebelum translate (untuk transparansi/audit)

---
## 1. Mount Google Drive & Install Dependencies

In [1]:
from google.colab import drive
drive.mount('/content/drive')
print('Google Drive berhasil di-mount!')

Mounted at /content/drive
Google Drive berhasil di-mount!


In [2]:
!pip install -q lingua-language-detector deep-translator
print('Dependencies installed ✓')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.3/170.3 MB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 4.1 MB/s eta 0:00:00
Dependencies installed ✓


In [3]:
import pandas as pd
import os
import re
import time
from datetime import datetime, timedelta
from lingua import Language, LanguageDetectorBuilder
from deep_translator import GoogleTranslator

# Whitelist bahasa yang realistis untuk hotel di Indonesia.
# Membatasi kandidat = akurasi naik signifikan untuk teks pendek.
ALLOWED_LANGS = [
    Language.INDONESIAN,
    Language.MALAY,        # sering tertukar dengan Indonesia
    Language.ENGLISH,
    Language.JAPANESE,
    Language.KOREAN,
    Language.CHINESE,
    Language.ARABIC,
    Language.TAGALOG,
    Language.THAI,
    Language.VIETNAMESE,
    Language.DUTCH,
    Language.FRENCH,
    Language.GERMAN,
    Language.SPANISH,
    Language.ITALIAN,
    Language.PORTUGUESE,
    Language.RUSSIAN,
    Language.HINDI,
    Language.TURKISH,
]

LANG_DETECTOR = (
    LanguageDetectorBuilder
    .from_languages(*ALLOWED_LANGS)
    .with_preloaded_language_models()
    .with_minimum_relative_distance(0.0)
    .build()
)

# Kode ISO 639-1 untuk mapping ke string singkat
LANG_TO_CODE = {
    Language.INDONESIAN: 'id',
    Language.MALAY: 'id',          # Malay diperlakukan sebagai Indonesia
    Language.ENGLISH: 'en',
    Language.JAPANESE: 'ja',
    Language.KOREAN: 'ko',
    Language.CHINESE: 'zh',
    Language.ARABIC: 'ar',
    Language.TAGALOG: 'tl',
    Language.THAI: 'th',
    Language.VIETNAMESE: 'vi',
    Language.DUTCH: 'nl',
    Language.FRENCH: 'fr',
    Language.GERMAN: 'de',
    Language.SPANISH: 'es',
    Language.ITALIAN: 'it',
    Language.PORTUGUESE: 'pt',
    Language.RUSSIAN: 'ru',
    Language.HINDI: 'hi',
    Language.TURKISH: 'tr',
}

print('All imports loaded ✓')

All imports loaded ✓


---
## 2. Konfigurasi

In [4]:
# ====== KONFIGURASI ======
BASE_DIR = '/content/drive/MyDrive/Scrap_Review_Santika'
OUTPUT_FILE = 'dataset_absa_santika.csv'

# Tanggal saat scraping Traveloka dilakukan
TRAVELOKA_SCRAPE_DATE = datetime(2026, 5, 27)

# Verifikasi folder
print(f'Base directory: {BASE_DIR}')
print(f'Traveloka scrape date: {TRAVELOKA_SCRAPE_DATE.strftime("%Y-%m-%d")}')
print()

if os.path.exists(BASE_DIR):
    print('Isi folder:')
    for item in sorted(os.listdir(BASE_DIR)):
        full_path = os.path.join(BASE_DIR, item)
        if os.path.isdir(full_path):
            files = [f for f in os.listdir(full_path) if f.endswith('.csv')]
            print(f'  📁 {item}/ ({len(files)} file CSV)')
            for f in sorted(files):
                size = os.path.getsize(os.path.join(full_path, f)) / 1024
                print(f'      - {f} ({size:.1f} KB)')
        else:
            print(f'  📄 {item}')
else:
    print(f'❌ Folder tidak ditemukan: {BASE_DIR}')
    print('   Pastikan path sudah benar!')

Base directory: /content/drive/MyDrive/Scrap_Review_Santika
Traveloka scrape date: 2026-05-27

Isi folder:
  📁 Agoda/ (6 file CSV)
      - Santika Bandung.csv (237.5 KB)
      - Santika Bekasi.csv (113.8 KB)
      - Santika Bogor.csv (806.6 KB)
      - Santika Cirebon.csv (613.7 KB)
      - Santika Depok.csv (189.6 KB)
      - Santika Tasikmalaya.csv (204.8 KB)
  📁 Tiket/ (6 file CSV)
      - Hotel Santika Bandung.csv (136.9 KB)
      - Hotel Santika Bekasi.csv (620.5 KB)
      - Hotel Santika Bogor tiket.csv (297.3 KB)
      - Hotel Santika Cirebon.csv (150.6 KB)
      - Hotel Santika Depok.csv (270.2 KB)
      - Hotel Santika Tasik.csv (109.8 KB)
  📁 Traveloka/ (6 file CSV)
      - Hotel Santika Bandung.csv (319.0 KB)
      - Hotel Santika Bogor.csv (417.7 KB)
      - Hotel Santika Cirebon.csv (242.3 KB)
      - Hotel Santika Depok.csv (438.0 KB)
      - Hotel Santika Megacity Bekasi.csv (459.7 KB)
      - Hotel Santika Tasikmalaya.csv (252.3 KB)
  📄 dataset_absa_santika.csv
  📄 data

---
## 3. Helper Functions

In [5]:
def normalize_hotel_name(filename):
    """Ekstrak & normalisasi nama hotel dari nama file CSV."""
    name = os.path.splitext(filename)[0]
    for prefix in ['Hotel ', 'hotel_', 'Hotel_']:
        if name.startswith(prefix):
            name = name[len(prefix):]
    name = name.strip()
    name_lower = name.lower()

    if 'bandung' in name_lower:
        return 'Hotel Santika Bandung'
    elif 'bekasi' in name_lower or 'megacity' in name_lower or 'mega city' in name_lower:
        return 'Hotel Santika Mega City Bekasi'
    elif 'bogor' in name_lower:
        return 'Hotel Santika Bogor'
    elif 'cirebon' in name_lower:
        return 'Hotel Santika Cirebon'
    elif 'depok' in name_lower:
        return 'Hotel Santika Depok'
    elif 'tasik' in name_lower:
        return 'Hotel Santika Tasikmalaya'
    else:
        return f'Hotel Santika {name}'


def convert_traveloka_date(relative_str, scrape_date):
    """Konversi 'Diulas X minggu/hari lalu' ke datetime."""
    if pd.isna(relative_str) or not isinstance(relative_str, str):
        return pd.NaT
    text = relative_str.strip().lower()

    match = re.match(r'diulas\s+(\d+)\s+minggu\s+lalu', text)
    if match:
        return scrape_date - timedelta(weeks=int(match.group(1)))

    match = re.match(r'diulas\s+(\d+)\s+hari\s+lalu', text)
    if match:
        return scrape_date - timedelta(days=int(match.group(1)))

    match = re.match(r'diulas\s+(\d+)\s+bulan\s+lalu', text)
    if match:
        return scrape_date - timedelta(days=int(match.group(1)) * 30)

    return pd.NaT


def parse_agoda_date(date_str):
    """Parse format Agoda: 'May 17, 2026'"""
    if pd.isna(date_str) or not isinstance(date_str, str):
        return pd.NaT
    try:
        return pd.to_datetime(date_str.strip(), format='%B %d, %Y')
    except:
        try:
            return pd.to_datetime(date_str.strip())
        except:
            return pd.NaT


def parse_tiket_date(date_str):
    """Parse format Tiket: '19 May 2026'"""
    if pd.isna(date_str) or not isinstance(date_str, str):
        return pd.NaT
    try:
        return pd.to_datetime(date_str.strip(), format='%d %b %Y')
    except:
        try:
            return pd.to_datetime(date_str.strip())
        except:
            return pd.NaT


def find_column(df, target_name):
    """Cari kolom (handle dash vs underscore)."""
    for col in df.columns:
        if col.lower().replace('-', '_') == target_name.lower():
            return col
    return None


print('Helper functions loaded ✓')

Helper functions loaded ✓


---
## 4. Baca & Proses Data per Platform

In [6]:
def process_platform(base_dir, platform, date_parser, scrape_date=None):
    """Baca semua CSV dari folder platform, return DataFrame."""
    folder = os.path.join(base_dir, platform)
    if not os.path.exists(folder):
        print(f'  ❌ Folder {platform} tidak ditemukan: {folder}')
        return pd.DataFrame()

    all_rows = []
    for fname in sorted(os.listdir(folder)):
        if not fname.endswith('.csv'):
            continue
        fpath = os.path.join(folder, fname)
        hotel_name = normalize_hotel_name(fname)
        df = pd.read_csv(fpath, encoding='utf-8-sig')

        text_col = find_column(df, 'teks_ulasan')
        date_col = find_column(df, 'tanggal')

        if text_col is None:
            print(f'  ⚠️  [SKIP] Kolom teks ulasan tidak ditemukan di {fname}: {df.columns.tolist()}')
            continue

        for _, row in df.iterrows():
            text = str(row.get(text_col, '')).strip() if pd.notna(row.get(text_col)) else ''
            date_raw = row.get(date_col, '') if date_col else ''

            if scrape_date:
                date_parsed = date_parser(date_raw, scrape_date)
            else:
                date_parsed = date_parser(date_raw)

            all_rows.append({
                'platform': platform,
                'hotel_name': hotel_name,
                'text_review': text,
                'date': date_parsed,
            })

        print(f'  ✅ {platform}/{fname}: {len(df)} baris → {hotel_name}')

    return pd.DataFrame(all_rows)


# --- Proses Agoda ---
print('📂 Membaca data Agoda...')
df_agoda = process_platform(BASE_DIR, 'Agoda', parse_agoda_date)
print(f'   Total Agoda: {len(df_agoda)} baris\n')

# --- Proses Tiket ---
print('📂 Membaca data Tiket...')
df_tiket = process_platform(BASE_DIR, 'Tiket', parse_tiket_date)
print(f'   Total Tiket: {len(df_tiket)} baris\n')

# --- Proses Traveloka ---
print('📂 Membaca data Traveloka (+ konversi tanggal relatif)...')
df_traveloka = process_platform(BASE_DIR, 'Traveloka', convert_traveloka_date, TRAVELOKA_SCRAPE_DATE)
print(f'   Total Traveloka: {len(df_traveloka)} baris')

📂 Membaca data Agoda...
  ✅ Agoda/Santika Bandung.csv: 829 baris → Hotel Santika Bandung
  ✅ Agoda/Santika Bekasi.csv: 438 baris → Hotel Santika Mega City Bekasi
  ✅ Agoda/Santika Bogor.csv: 2479 baris → Hotel Santika Bogor
  ✅ Agoda/Santika Cirebon.csv: 1686 baris → Hotel Santika Cirebon
  ✅ Agoda/Santika Depok.csv: 767 baris → Hotel Santika Depok
  ✅ Agoda/Santika Tasikmalaya.csv: 743 baris → Hotel Santika Tasikmalaya
   Total Agoda: 6942 baris

📂 Membaca data Tiket...
  ✅ Tiket/Hotel Santika Bandung.csv: 1000 baris → Hotel Santika Bandung
  ✅ Tiket/Hotel Santika Bekasi.csv: 2083 baris → Hotel Santika Mega City Bekasi
  ✅ Tiket/Hotel Santika Bogor tiket.csv: 1972 baris → Hotel Santika Bogor
  ✅ Tiket/Hotel Santika Cirebon.csv: 1036 baris → Hotel Santika Cirebon
  ✅ Tiket/Hotel Santika Depok.csv: 2083 baris → Hotel Santika Depok
  ✅ Tiket/Hotel Santika Tasik.csv: 810 baris → Hotel Santika Tasikmalaya
   Total Tiket: 8984 baris

📂 Membaca data Traveloka (+ konversi tanggal relatif)...


---
## 5. Gabung & Cleaning Awal

In [7]:
# Gabungkan semua platform
df_all = pd.concat([df_agoda, df_tiket, df_traveloka], ignore_index=True)
print(f'Total baris sebelum cleaning: {len(df_all):,}')

# Hapus review dengan teks kosong
df_all = df_all[df_all['text_review'].str.strip().astype(bool)]
print(f'Setelah hapus review kosong:  {len(df_all):,}')

# Hapus baris tanpa tanggal
before = len(df_all)
df_all = df_all.dropna(subset=['date'])
print(f'Setelah hapus tanggal kosong: {len(df_all):,} (hapus {before - len(df_all):,})')

# Reset index
df_all = df_all.reset_index(drop=True)

print(f'\nDataset siap untuk deteksi bahasa: {len(df_all):,} review')

Total baris sebelum cleaning: 24,789
Setelah hapus review kosong:  20,989
Setelah hapus tanggal kosong: 17,868 (hapus 3,121)

Dataset siap untuk deteksi bahasa: 17,868 review


---
## 6. Deteksi Bahasa (lingua-language-detector)

Library `lingua` dipilih karena:
- **Akurasi tinggi pada teks pendek** (review hotel sering hanya 1-2 kalimat)
- **Whitelist bahasa** mencegah misdetection ke bahasa eksotis (Welsh, Estonia, Somalia, dll) yang sebelumnya terjadi dengan `langdetect`
- **Confidence scoring** memungkinkan fallback ke `unknown` saat tidak yakin

Output kolom `original_language` (kode ISO 639-1):
- `id` = Indonesia (termasuk Malay, karena sangat mirip)
- `en` = English
- `ja, ko, zh, ar, tl, th, vi, nl, fr, de, es, it, pt, ru, hi, tr` = bahasa lain
- `unknown` = teks terlalu pendek atau confidence rendah

In [8]:
def detect_language(text):
    """Deteksi bahasa pakai lingua. Return kode ISO 639-1 atau 'unknown'."""
    if not isinstance(text, str):
        return 'unknown'
    text_clean = text.strip()
    if len(text_clean) < 10:
        return 'unknown'
    try:
        # Cek confidence — kalau ragu, return 'unknown'
        confidence = LANG_DETECTOR.compute_language_confidence_values(text_clean)
        if not confidence:
            return 'unknown'
        top = confidence[0]
        # Skor terlalu rendah → tidak yakin
        if top.value < 0.50:
            return 'unknown'
        return LANG_TO_CODE.get(top.language, 'unknown')
    except Exception:
        return 'unknown'


print('Mendeteksi bahasa setiap review pakai lingua...')
print('(Estimasi: ~1-2 menit untuk ~17.000 review)\n')

start_time = time.time()
df_all['original_language'] = df_all['text_review'].apply(detect_language)
elapsed = time.time() - start_time

print(f'Deteksi bahasa selesai dalam {elapsed:.1f} detik\n')

# Distribusi bahasa
lang_dist = df_all['original_language'].value_counts()
print('Distribusi bahasa:')
LABEL_MAP = {
    'id': 'Indonesia', 'en': 'English', 'ja': 'Japanese', 'ko': 'Korean',
    'zh': 'Chinese', 'ar': 'Arabic', 'tl': 'Tagalog', 'th': 'Thai',
    'vi': 'Vietnamese', 'nl': 'Dutch', 'fr': 'French', 'de': 'German',
    'es': 'Spanish', 'it': 'Italian', 'pt': 'Portuguese', 'ru': 'Russian',
    'hi': 'Hindi', 'tr': 'Turkish', 'unknown': 'Tidak terdeteksi/terlalu pendek',
}
for lang, count in lang_dist.items():
    pct = count / len(df_all) * 100
    label = LABEL_MAP.get(lang, lang)
    print(f'  {lang:>7s} ({label}): {count:,} review ({pct:.1f}%)')

# Breakdown per platform
print('\nBreakdown per platform:')
display(pd.crosstab(df_all['platform'], df_all['original_language'], margins=True))

Mendeteksi bahasa setiap review pakai lingua...
(Estimasi: ~1-2 menit untuk ~17.000 review)

Deteksi bahasa selesai dalam 13.5 detik

Distribusi bahasa:
       id (Indonesia): 13,855 review (77.5%)
       en (English): 2,275 review (12.7%)
  unknown (Tidak terdeteksi/terlalu pendek): 1,563 review (8.7%)
       ja (Japanese): 59 review (0.3%)
       ko (Korean): 28 review (0.2%)
       tl (Tagalog): 21 review (0.1%)
       nl (Dutch): 17 review (0.1%)
       fr (French): 13 review (0.1%)
       ar (Arabic): 12 review (0.1%)
       de (German): 10 review (0.1%)
       zh (Chinese): 6 review (0.0%)
       es (Spanish): 3 review (0.0%)
       th (Thai): 2 review (0.0%)
       it (Italian): 2 review (0.0%)
       ru (Russian): 2 review (0.0%)

Breakdown per platform:


original_language,ar,de,en,es,fr,id,it,ja,ko,nl,ru,th,tl,unknown,zh,All
platform,,,,,,,,,,,,,,,,
Agoda,11,10,2050,0,11,3369,1,56,28,14,2,2,3,678,6,6241
Tiket,1,0,218,3,2,2080,0,3,0,3,0,0,12,442,0,2764
Traveloka,0,0,7,0,0,8406,1,0,0,0,0,0,6,443,0,8863
All,12,10,2275,3,13,13855,2,59,28,17,2,2,21,1563,6,17868


---
## 7. Translate Semua Review Non-Indonesia → Indonesia

Menggunakan `deep-translator` (Google Translate gratis) untuk menerjemahkan semua review non-id.

**Strategi:**
- Translate semua bahasa selain `id` dan `unknown`: `en, ja, ko, zh, ar, tl, dst.`
- Review `id` (Indonesia) tetap dipertahankan apa adanya
- Review `unknown` (terlalu pendek atau tidak yakin) tetap dipertahankan, tidak ditranslate
- **Teks asli disimpan ke kolom `text_review_original`** untuk audit/transparansi
- Batch processing dengan retry & rate limiting
- Checkpoint disimpan ke Google Drive agar tahan disconnect

In [9]:
# Backup teks asli SEBELUM translate (untuk transparansi)
df_all['text_review_original'] = df_all['text_review'].copy()

# Identifikasi review yang perlu ditranslate: semua selain id & unknown
LANGS_TO_TRANSLATE = [l for l in df_all['original_language'].unique()
                      if l not in ('id', 'unknown')]
needs_translation = df_all['original_language'].isin(LANGS_TO_TRANSLATE)
total_to_translate = needs_translation.sum()

print(f'Review yang perlu ditranslate (non-id → id): {total_to_translate:,}')
print(f'Bahasa yang akan ditranslate: {sorted(LANGS_TO_TRANSLATE)}')
print(f'Review bahasa Indonesia (skip): {(df_all["original_language"] == "id").sum():,}')
print(f'Review unknown (skip): {(df_all["original_language"] == "unknown").sum():,}')

# Breakdown jumlah per bahasa yang akan ditranslate
print('\nDistribusi review yang akan ditranslate:')
print(df_all[needs_translation]['original_language'].value_counts().to_string())

Review yang perlu ditranslate (non-id → id): 2,450
Bahasa yang akan ditranslate: ['ar', 'de', 'en', 'es', 'fr', 'it', 'ja', 'ko', 'nl', 'ru', 'th', 'tl', 'zh']
Review bahasa Indonesia (skip): 13,855
Review unknown (skip): 1,563

Distribusi review yang akan ditranslate:
original_language
en    2275
ja      59
ko      28
tl      21
nl      17
fr      13
ar      12
de      10
zh       6
es       3
th       2
it       2
ru       2


In [10]:
# Path checkpoint di Google Drive
CHECKPOINT_PATH = os.path.join(BASE_DIR, '_translation_checkpoint.csv')


def translate_one(text, src_lang, target='id'):
    """Translate single text. Auto-detect source jika src_lang gagal."""
    if not text or len(str(text).strip()) == 0:
        return text
    text = str(text)[:4900]  # deep-translator max 5000 chars

    try:
        # Coba dengan source lang yang terdeteksi
        translator = GoogleTranslator(source=src_lang, target=target)
        result = translator.translate(text)
        return result if result else text
    except Exception:
        # Fallback ke auto-detect
        try:
            translator = GoogleTranslator(source='auto', target=target)
            result = translator.translate(text)
            return result if result else text
        except Exception:
            return text  # Gagal total → kembalikan teks asli


def translate_indices(df, indices, batch_delay=0.3):
    """Translate review pada indices tertentu dari df. Return dict {idx: translated}."""
    results = {}
    total = len(indices)

    for i, idx in enumerate(indices, 1):
        text = df.at[idx, 'text_review']
        src_lang = df.at[idx, 'original_language']

        # Mapping ke kode bahasa Google Translate
        # Sebagian besar kode ISO 639-1 sudah kompatibel dengan Google Translate
        gt_src = src_lang if src_lang not in ('unknown',) else 'auto'

        translated = translate_one(text, gt_src, target='id')
        results[idx] = translated

        # Progress
        if i % 10 == 0 or i == total:
            pct = i / total * 100
            print(f'  Translated: {i:,}/{total:,} ({pct:.1f}%)', end='\r')

        # Rate limiting
        time.sleep(batch_delay)

    print()  # New line setelah progress
    return results


# === MULAI TRANSLATE ===
if total_to_translate > 0:
    en_indices = df_all[needs_translation].index.tolist()

    # Cek checkpoint
    if os.path.exists(CHECKPOINT_PATH):
        print(f'📂 Checkpoint ditemukan, melanjutkan dari checkpoint...')
        df_checkpoint = pd.read_csv(CHECKPOINT_PATH, encoding='utf-8-sig')
        translated_map = dict(zip(df_checkpoint['index'].astype(int),
                                  df_checkpoint['translated']))
        remaining_indices = [idx for idx in en_indices if idx not in translated_map]
        print(f'   Sudah ditranslate: {len(translated_map):,}')
        print(f'   Tersisa: {len(remaining_indices):,}')
    else:
        translated_map = {}
        remaining_indices = en_indices

    if len(remaining_indices) > 0:
        print(f'\n🔄 Menerjemahkan {len(remaining_indices):,} review (non-id → id)...')
        print(f'   Estimasi waktu: ~{len(remaining_indices) * 0.5 / 60:.0f} menit')
        print(f'   Checkpoint disimpan setiap 200 review\n')

        # Translate dalam chunk dengan auto-checkpoint setiap 200 review
        CHECKPOINT_EVERY = 200
        for chunk_start in range(0, len(remaining_indices), CHECKPOINT_EVERY):
            chunk = remaining_indices[chunk_start:chunk_start + CHECKPOINT_EVERY]
            chunk_results = translate_indices(df_all, chunk, batch_delay=0.3)
            translated_map.update(chunk_results)

            # Auto-save checkpoint
            df_ckpt = pd.DataFrame([
                {'index': k, 'translated': v} for k, v in translated_map.items()
            ])
            df_ckpt.to_csv(CHECKPOINT_PATH, index=False, encoding='utf-8-sig')
            print(f'  💾 Checkpoint disimpan ({len(translated_map):,} total)')

    # Apply translations ke kolom text_review (kolom text_review_original tetap asli)
    for idx, translated in translated_map.items():
        if idx in df_all.index:
            df_all.at[idx, 'text_review'] = translated

    print(f'\n✅ Translate selesai! {len(translated_map):,} review berhasil diterjemahkan.')
else:
    print('Tidak ada review non-Indonesia, skip translate.')


🔄 Menerjemahkan 2,450 review (non-id → id)...
   Estimasi waktu: ~20 menit
   Checkpoint disimpan setiap 200 review


  💾 Checkpoint disimpan (200 total)

  💾 Checkpoint disimpan (400 total)

  💾 Checkpoint disimpan (600 total)

  💾 Checkpoint disimpan (800 total)

  💾 Checkpoint disimpan (1,000 total)

  💾 Checkpoint disimpan (1,200 total)

  💾 Checkpoint disimpan (1,400 total)

  💾 Checkpoint disimpan (1,600 total)

  💾 Checkpoint disimpan (1,800 total)

  💾 Checkpoint disimpan (2,000 total)

  💾 Checkpoint disimpan (2,200 total)

  💾 Checkpoint disimpan (2,400 total)

  💾 Checkpoint disimpan (2,450 total)

✅ Translate selesai! 2,450 review berhasil diterjemahkan.


---
## 8. Verifikasi Hasil Translate

Cek sample review (asli vs hasil translate) untuk berbagai bahasa.

In [11]:
# Tampilkan sample hasil translate per bahasa
LANGS_TO_TRANSLATE_SET = [l for l in df_all['original_language'].unique()
                          if l not in ('id', 'unknown')]

if LANGS_TO_TRANSLATE_SET:
    print('Sample hasil translate per bahasa:')
    print('=' * 80)

    for lang in sorted(LANGS_TO_TRANSLATE_SET):
        subset = df_all[df_all['original_language'] == lang].head(2)
        if len(subset) == 0:
            continue
        print(f'\n--- {lang.upper()} ({len(df_all[df_all["original_language"] == lang])} review) ---')
        for _, row in subset.iterrows():
            orig = row['text_review_original'][:120]
            trans = row['text_review'][:120]
            print(f'  ASLI    : {orig}')
            print(f'  TRANSLATE: {trans}')
            print()
else:
    print('Tidak ada review yang perlu ditranslate.')

Sample hasil translate per bahasa:

--- AR (12 review) ---
  ASLI    : ممتاز في النظافة والاقامة هذة الاقامة الثانية وسيكرر الاقامة الثالثة عما قريب ولكن نطلب من ادارة الفندق ان تمنحني سعرمخف
  TRANSLATE: Sangat baik dalam kebersihan dan akomodasi. Ini adalah kunjungan kedua dan saya akan segera mengulangi kunjungan ketiga,

  ASLI    : فندق سانتيكا فندق رائع وجميل خاصة موقعة عند مركزالتجاري بوتاني جميل جدا وموقعة ايضا في مدينة بوقور مدينة رائعة في الاجوا
  TRANSLATE: Santika Hotel merupakan hotel yang indah dan indah, terutama lokasinya yang dekat dengan Bhutani Mall. Sangat indah dan 


--- DE (10 review) ---
  ASLI    : Gute Lage zum Erkunden von Bogor, direkte Anbindung an Botani Mall. Frühstück abwechslungsreich.
  TRANSLATE: Lokasi bagus untuk menjelajah Bogor, akses langsung ke Botani Mall. Sarapan bervariasi.

  ASLI    : Gutes Preis Leistungsverhältnis, einfache Ausstattung, freundliches Personal, neues Hotel, wirklich nur 3 Sterne, Zugang
  TRANSLATE: Rasio harga-kinerja bagu

---
## 9. Finalisasi Dataset

In [12]:
# Tambahkan review_id
df_all = df_all.reset_index(drop=True)
df_all.insert(0, 'review_id', range(1, len(df_all) + 1))

# Format tanggal → YYYY-MM-DD
df_all['date'] = pd.to_datetime(df_all['date']).dt.strftime('%Y-%m-%d')

# Urutkan kolom
df_all = df_all[[
    'review_id', 'platform', 'hotel_name',
    'text_review', 'text_review_original',
    'date', 'original_language',
]]

print(f'Dataset final: {len(df_all):,} review')
print(f'Kolom: {df_all.columns.tolist()}')
df_all.head(10)

Dataset final: 17,868 review
Kolom: ['review_id', 'platform', 'hotel_name', 'text_review', 'text_review_original', 'date', 'original_language']


,review_id,platform,hotel_name,text_review,text_review_original,date,original_language
0,1,Agoda,Hotel Santika Bandung,Saya selalu menginap di Santika bila di Bandun...,I always stay at Santika when in Bandung. Many...,2026-05-17,en
1,2,Agoda,Hotel Santika Bandung,"Bisa jalan kaki langsung ke BIP, karena lokasi...","You can walk directly to BIP, because the loca...",2026-05-11,en
2,3,Agoda,Hotel Santika Bandung,Lokasinya strategis di pusat kota. Makanan saa...,The location is strategic at the city centre. ...,2026-05-05,en
3,4,Agoda,Hotel Santika Bandung,Hotel ini agak ketinggalan jaman. Stafnya rama...,The hotel is a bit outdated. The staff are fri...,2026-03-31,en
4,5,Agoda,Hotel Santika Bandung,"Pengalaman menginap yang sangat berkesan, pros...","A very memorable stay experience, the check-in...",2026-03-26,en
5,6,Agoda,Hotel Santika Bandung,"Berlokasi strategis di jantung kota Bandung, d...","Strategically located in the heart of Bandung,...",2026-03-25,en
6,7,Agoda,Hotel Santika Bandung,Hotel yg baik dengan staff yg sangat ramah . S...,Hotel yg baik dengan staff yg sangat ramah . S...,2026-03-24,id
7,8,Agoda,Hotel Santika Bandung,Trims kemaren dapat kasur per lebih kokoh tida...,Trims kemaren dapat kasur per lebih kokoh tida...,2026-03-20,id
8,9,Agoda,Hotel Santika Bandung,"Dari luar mungkin terlihat tua, namun kamarnya...","It may look old from the outside, but the room...",2026-03-17,en
9,10,Agoda,Hotel Santika Bandung,Lokasinya berada di antara BIP dan Yogya Riau ...,The location is between BIP and Yogya Riau Jun...,2026-03-02,en


---
## 10. Ringkasan Dataset

In [13]:
print('=' * 55)
print('  RINGKASAN DATASET FINAL')
print('=' * 55)
print(f'Total review: {len(df_all):,}')
print(f'Rentang tanggal: {df_all["date"].min()} s/d {df_all["date"].max()}')

print('\n--- Per Platform ---')
display(df_all.groupby('platform').size().to_frame('jumlah_review').reset_index())

print('\n--- Per Hotel ---')
display(df_all.groupby('hotel_name').size().to_frame('jumlah_review').reset_index())

print('\n--- Bahasa Asli Review ---')
display(df_all.groupby('original_language').size().to_frame('jumlah').reset_index())

print('\n--- Crosstab Platform x Hotel ---')
display(pd.crosstab(df_all['hotel_name'], df_all['platform'], margins=True))

  RINGKASAN DATASET FINAL
Total review: 17,868
Rentang tanggal: 2010-08-02 s/d 2026-05-25

--- Per Platform ---


,platform,jumlah_review
0,Agoda,6241
1,Tiket,2764
2,Traveloka,8863



--- Per Hotel ---


,hotel_name,jumlah_review
0,Hotel Santika Bandung,2783
1,Hotel Santika Bogor,4291
2,Hotel Santika Cirebon,3031
3,Hotel Santika Depok,2857
4,Hotel Santika Mega City Bekasi,2957
5,Hotel Santika Tasikmalaya,1949



--- Bahasa Asli Review ---


,original_language,jumlah
0,ar,12
1,de,10
2,en,2275
3,es,3
4,fr,13
5,id,13855
6,it,2
7,ja,59
8,ko,28
9,nl,17



--- Crosstab Platform x Hotel ---


platform,Agoda,Tiket,Traveloka,All
hotel_name,,,,
Hotel Santika Bandung,775,671,1337,2783
Hotel Santika Bogor,2408,281,1602,4291
Hotel Santika Cirebon,1650,315,1066,3031
Hotel Santika Depok,716,221,1920,2857
Hotel Santika Mega City Bekasi,0,1147,1810,2957
Hotel Santika Tasikmalaya,692,129,1128,1949
All,6241,2764,8863,17868


---
## 11. Export CSV ke Google Drive

In [14]:
# Simpan ke Google Drive
output_path = os.path.join(BASE_DIR, OUTPUT_FILE)
df_all.to_csv(output_path, index=False, encoding='utf-8-sig')

file_size = os.path.getsize(output_path) / 1024 / 1024
print(f'✅ Dataset berhasil disimpan ke Google Drive!')
print(f'   Path: {output_path}')
print(f'   Ukuran: {file_size:.2f} MB')
print(f'   Jumlah baris: {len(df_all):,}')
print(f'   Kolom: {df_all.columns.tolist()}')

# Hapus checkpoint jika sudah selesai
if os.path.exists(CHECKPOINT_PATH):
    os.remove(CHECKPOINT_PATH)
    print(f'\n🗑️ Checkpoint dihapus (sudah tidak diperlukan)')

✅ Dataset berhasil disimpan ke Google Drive!
   Path: /content/drive/MyDrive/Scrap_Review_Santika/dataset_absa_santika.csv
   Ukuran: 6.01 MB
   Jumlah baris: 17,868
   Kolom: ['review_id', 'platform', 'hotel_name', 'text_review', 'text_review_original', 'date', 'original_language']

🗑️ Checkpoint dihapus (sudah tidak diperlukan)


---
## 12. (Opsional) Download ke Lokal

In [20]:
from google.colab import files
files.download(output_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

---
## 13. PATCH — Refine Deteksi Bahasa & Cleanup

> **Cell ini standalone** — bisa dijalankan tanpa run ulang cell-cell sebelumnya. Cukup pastikan `dataset_absa_santika.csv` sudah ada di Google Drive dari run sebelumnya.

**Tujuan patch ini:**
1. **Turunkan threshold confidence** dari 0.50 → 0.30 supaya review pendek berbahasa Indonesia tidak salah dilabeli `unknown`
2. **Heuristic kata kunci Indonesia** — jika teks pendek mengandung kata khas ID, reassign ke `id`
3. **Rollback terjemahan yang salah** — review yang sebelumnya dilabeli `tl` (Tagalog) tapi sebenarnya Indonesia campur Inggris, balikkan `text_review` ke aslinya
4. **Translate ulang** review yang baru terdeteksi non-id non-en (kalau ada)

**Output:** `dataset_absa_santika.csv` yang sama, di-overwrite dengan label & teks yang sudah diperbaiki.

In [21]:
# ====== PATCH CELL 1: Load dataset & re-setup detector ======
import pandas as pd
import os
import re
import time
from lingua import Language, LanguageDetectorBuilder

try:
    BASE_DIR
except NameError:
    BASE_DIR = '/content/drive/MyDrive/Scrap_Review_Santika'

INPUT_FILE = os.path.join(BASE_DIR, 'dataset_absa_santika.csv')
df_all = pd.read_csv(INPUT_FILE, encoding='utf-8-sig')

print(f'Loaded: {len(df_all):,} review dari {INPUT_FILE}')
print(f'Kolom: {df_all.columns.tolist()}')
print()
print('Distribusi bahasa SEBELUM patch:')
print(df_all['original_language'].value_counts().to_string())

Loaded: 17,868 review dari /content/drive/MyDrive/Scrap_Review_Santika/dataset_absa_santika.csv
Kolom: ['review_id', 'platform', 'hotel_name', 'text_review', 'text_review_original', 'date', 'original_language']

Distribusi bahasa SEBELUM patch:
original_language
id         13855
en          2275
unknown     1563
ja            59
ko            28
tl            21
nl            17
fr            13
ar            12
de            10
zh             6
es             3
th             2
it             2
ru             2


In [22]:
# ====== PATCH CELL 2: Re-detect bahasa dengan threshold lebih longgar + heuristic ======

# Setup detector ulang (standalone, kalau runtime sudah restart)
ALLOWED_LANGS = [
    Language.INDONESIAN, Language.MALAY, Language.ENGLISH,
    Language.JAPANESE, Language.KOREAN, Language.CHINESE,
    Language.ARABIC, Language.TAGALOG, Language.THAI,
    Language.VIETNAMESE, Language.DUTCH, Language.FRENCH,
    Language.GERMAN, Language.SPANISH, Language.ITALIAN,
    Language.PORTUGUESE, Language.RUSSIAN, Language.HINDI,
    Language.TURKISH,
]
LANG_DETECTOR = LanguageDetectorBuilder.from_languages(*ALLOWED_LANGS).with_preloaded_language_models().build()
LANG_TO_CODE = {
    Language.INDONESIAN: 'id', Language.MALAY: 'id', Language.ENGLISH: 'en',
    Language.JAPANESE: 'ja', Language.KOREAN: 'ko', Language.CHINESE: 'zh',
    Language.ARABIC: 'ar', Language.TAGALOG: 'tl', Language.THAI: 'th',
    Language.VIETNAMESE: 'vi', Language.DUTCH: 'nl', Language.FRENCH: 'fr',
    Language.GERMAN: 'de', Language.SPANISH: 'es', Language.ITALIAN: 'it',
    Language.PORTUGUESE: 'pt', Language.RUSSIAN: 'ru', Language.HINDI: 'hi',
    Language.TURKISH: 'tr',
}

# Kata kunci khas bahasa Indonesia. Jika teks mengandung >=2 kata ini, treat as 'id'.
ID_KEYWORDS = {
    'yang', 'saya', 'tidak', 'sangat', 'dengan', 'untuk', 'sudah', 'juga',
    'bagus', 'enak', 'ramah', 'bersih', 'nyaman', 'kamar', 'hotel', 'staff',
    'staf', 'pelayanan', 'sarapan', 'kolam', 'pemandangan', 'lokasi', 'mantap',
    'menyenangkan', 'membantu', 'gak', 'aja', 'kayak', 'banget', 'lagi', 'kalo',
    'kalau', 'biasa', 'cukup', 'kurang', 'lebih', 'pernah', 'banyak', 'sekali',
    'kembali', 'baik', 'baru', 'lama', 'tempat', 'depan', 'dekat', 'jauh',
    'breakfast',  # umum dipakai di review hotel Indonesia
}

def is_indonesian_by_keywords(text, min_hits=2):
    """Cek apakah teks 'cukup Indonesia' berdasarkan kata kunci."""
    if not isinstance(text, str):
        return False
    words = re.findall(r'\b[a-zA-Z]+\b', text.lower())
    hits = sum(1 for w in words if w in ID_KEYWORDS)
    return hits >= min_hits


def detect_language_v2(text):
    """Re-detect bahasa dengan threshold 0.30 + heuristic kata kunci Indonesia."""
    if not isinstance(text, str):
        return 'unknown'
    text_clean = text.strip()
    if len(text_clean) < 10:
        # Untuk teks sangat pendek, cek apakah kata Indonesia
        if is_indonesian_by_keywords(text_clean, min_hits=1):
            return 'id'
        return 'unknown'

    try:
        confidence = LANG_DETECTOR.compute_language_confidence_values(text_clean)
        if not confidence:
            # Fallback ke heuristic
            return 'id' if is_indonesian_by_keywords(text_clean) else 'unknown'

        top = confidence[0]
        top_code = LANG_TO_CODE.get(top.language, 'unknown')

        # Confidence cukup tinggi → trust hasilnya
        if top.value >= 0.30:
            # SPECIAL CASE: kalau hasil 'tl' (Tagalog), cross-check dengan kata Indonesia
            # Karena Tagalog & Indonesia sering campur Inggris, mudah tertukar
            if top_code == 'tl' and is_indonesian_by_keywords(text_clean):
                return 'id'
            return top_code

        # Confidence rendah → fallback ke heuristic
        if is_indonesian_by_keywords(text_clean):
            return 'id'

        # Kalau hasil top adalah 'id' meskipun confidence rendah, masih reasonable
        if top_code == 'id' and top.value >= 0.15:
            return 'id'

        return 'unknown'
    except Exception:
        return 'id' if is_indonesian_by_keywords(text_clean) else 'unknown'


# Re-detect berdasarkan text_review_original (teks asli sebelum translate)
print('Re-detect bahasa (threshold 0.30 + heuristic kata kunci ID)...')
print(f'Source kolom: text_review_original (teks asli, bukan hasil translate)\n')

start = time.time()
df_all['original_language_new'] = df_all['text_review_original'].apply(detect_language_v2)
elapsed = time.time() - start

print(f'Selesai dalam {elapsed:.1f} detik\n')

# Bandingkan
print('Distribusi bahasa SETELAH patch:')
print(df_all['original_language_new'].value_counts().to_string())
print()

# Berapa label yang berubah
changed = (df_all['original_language'] != df_all['original_language_new']).sum()
print(f'Total label yang BERUBAH: {changed:,} review')
print()
print('Detail perubahan label:')
change_table = pd.crosstab(df_all['original_language'], df_all['original_language_new'], margins=True)
display(change_table)

Re-detect bahasa (threshold 0.30 + heuristic kata kunci ID)...
Source kolom: text_review_original (teks asli, bukan hasil translate)

Selesai dalam 13.2 detik

Distribusi bahasa SETELAH patch:
original_language_new
id         14801
en          2468
unknown      385
ja            59
ko            28
tl            28
fr            22
nl            20
de            14
es            12
ar            12
zh             6
pt             4
th             2
it             2
ru             2
tr             2
vi             1

Total label yang BERUBAH: 1,193 review

Detail perubahan label:


original_language_new,ar,de,en,es,fr,id,it,ja,ko,nl,pt,ru,th,tl,tr,unknown,vi,zh,All
original_language,,,,,,,,,,,,,,,,,,,
ar,12,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,12
de,0,10,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,10
en,0,0,2275,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2275
es,0,0,0,3,0,0,0,0,0,0,0,0,0,0,0,0,0,0,3
fr,0,0,0,0,13,0,0,0,0,0,0,0,0,0,0,0,0,0,13
id,0,0,0,0,0,13855,0,0,0,0,0,0,0,0,0,0,0,0,13855
it,0,0,0,0,0,0,2,0,0,0,0,0,0,0,0,0,0,0,2
ja,0,0,0,0,0,0,0,59,0,0,0,0,0,0,0,0,0,0,59
ko,0,0,0,0,0,0,0,0,28,0,0,0,0,0,0,0,0,0,28


In [23]:
# ====== PATCH CELL 3: Rollback text_review untuk row yang sekarang id/unknown ======
# Kalau label baru = 'id' atau 'unknown' tapi text_review sudah ditranslate (beda dari original),
# itu artinya translate sebelumnya salah → balikkan ke text_review_original

# Apply label baru
df_all['original_language'] = df_all['original_language_new']
df_all = df_all.drop(columns=['original_language_new'])

# Rollback
ROLLBACK_LANGS = ['id', 'unknown']
needs_rollback = (
    df_all['original_language'].isin(ROLLBACK_LANGS)
    & (df_all['text_review'] != df_all['text_review_original'])
)
n_rollback = needs_rollback.sum()
print(f'Review yang di-rollback (text_review → text_review_original): {n_rollback:,}')

if n_rollback > 0:
    print('\nSample rollback (sebelum rollback):')
    sample = df_all[needs_rollback].head(5)
    for _, r in sample.iterrows():
        print(f'  [{r["original_language"]}] (sebelum) translate: "{r["text_review"][:80]}"')
        print(f'         (sesudah) original : "{r["text_review_original"][:80]}"')
        print()

    df_all.loc[needs_rollback, 'text_review'] = df_all.loc[needs_rollback, 'text_review_original']
    print(f'✅ {n_rollback:,} review berhasil di-rollback ke teks asli')
else:
    print('Tidak ada yang perlu di-rollback.')

Review yang di-rollback (text_review → text_review_original): 15

Sample rollback (sebelum rollback):
  [id] (sebelum) translate: "Nyaman, bersih, staf ramah dan membantu, sarapannya enak, next time saya akan se"
         (sesudah) original : "Nyaman, bersih, staff ramah dan helpful, breakfast enak, next time ke sini lagi"

  [id] (sebelum) translate: "Ini hotel yang bagus.. stafnya sangat baik... makanannya enak... cuman memang ba"
         (sesudah) original : "Bagus hotel nya.. staff sangat baik... makanan enak... cuman emang bangunan tua "

  [id] (sebelum) translate: "Sayang sekali hotel budget 500rb tapi bukan smart TV 😭 Tapi secara keseluruhan o"
         (sesudah) original : "Sayang bgt budget hotel 500rb tapi bukan smart TV 😭 But overall okay, makanan ju"

  [id] (sebelum) translate: "Menunya enak. Area parkir terpelihara dengan baik. Harga murah hati. Interior mi"
         (sesudah) original : "Delicious menu. The parking area is well maintained. Generous pricing. Beautiful"


In [24]:
# ====== PATCH CELL 4: Translate review yang BARU terdeteksi non-id (kalau ada) ======
# Skenario: review yang sebelumnya 'unknown' ternyata bahasa asing dan belum pernah ditranslate

from deep_translator import GoogleTranslator

LANGS_TO_TRANSLATE = [l for l in df_all['original_language'].unique()
                      if l not in ('id', 'unknown')]

# Cari row yang labelnya non-id non-unknown TAPI text_review masih sama dengan original (belum ditranslate)
needs_new_translate = (
    df_all['original_language'].isin(LANGS_TO_TRANSLATE)
    & (df_all['text_review'] == df_all['text_review_original'])
)
n_new = needs_new_translate.sum()
print(f'Review baru yang perlu ditranslate: {n_new:,}')
print(f'(Bahasa: {df_all[needs_new_translate]["original_language"].value_counts().to_dict()})')

if n_new > 0:
    print(f'\n🔄 Menerjemahkan {n_new:,} review baru...')

    def translate_one(text, src, target='id'):
        if not text or len(str(text).strip()) == 0:
            return text
        text = str(text)[:4900]
        try:
            return GoogleTranslator(source=src, target=target).translate(text) or text
        except Exception:
            try:
                return GoogleTranslator(source='auto', target=target).translate(text) or text
            except Exception:
                return text

    indices_to_translate = df_all[needs_new_translate].index.tolist()
    for i, idx in enumerate(indices_to_translate, 1):
        text = df_all.at[idx, 'text_review_original']
        src = df_all.at[idx, 'original_language']
        translated = translate_one(text, src, target='id')
        df_all.at[idx, 'text_review'] = translated

        if i % 10 == 0 or i == n_new:
            print(f'  Progress: {i}/{n_new} ({i/n_new*100:.0f}%)', end='\r')
        time.sleep(0.3)
    print(f'\n✅ {n_new:,} review baru berhasil diterjemahkan')
else:
    print('Tidak ada review baru yang perlu ditranslate.')

Review baru yang perlu ditranslate: 250
(Bahasa: {'en': 194, 'tl': 22, 'fr': 9, 'es': 9, 'de': 4, 'pt': 4, 'nl': 3, 'zh': 2, 'tr': 2, 'vi': 1})

🔄 Menerjemahkan 250 review baru...

✅ 250 review baru berhasil diterjemahkan


In [25]:
# ====== PATCH CELL 5: Save & Ringkasan ======

# Re-format final
df_all = df_all[[
    'review_id', 'platform', 'hotel_name',
    'text_review', 'text_review_original',
    'date', 'original_language',
]]

# Save ulang ke file yang sama (overwrite)
output_path = os.path.join(BASE_DIR, 'dataset_absa_santika.csv')
df_all.to_csv(output_path, index=False, encoding='utf-8-sig')

print('=' * 60)
print('  PATCH SELESAI')
print('=' * 60)
print(f'✅ Dataset di-update: {output_path}')
print(f'   Total review: {len(df_all):,}')
print(f'   Ukuran: {os.path.getsize(output_path) / 1024 / 1024:.2f} MB')
print()
print('Distribusi bahasa FINAL:')
print(df_all['original_language'].value_counts().to_string())
print()

n_translated = (df_all['text_review'] != df_all['text_review_original']).sum()
print(f'Review yang ditranslate: {n_translated:,}')
print(f'Review bahasa asli (id/unknown): {len(df_all) - n_translated:,}')

  PATCH SELESAI
✅ Dataset di-update: /content/drive/MyDrive/Scrap_Review_Santika/dataset_absa_santika.csv
   Total review: 17,868
   Ukuran: 6.01 MB

Distribusi bahasa FINAL:
original_language
id         14801
en          2468
unknown      385
ja            59
ko            28
tl            28
fr            22
nl            20
de            14
es            12
ar            12
zh             6
pt             4
th             2
it             2
ru             2
tr             2
vi             1

Review yang ditranslate: 2,671
Review bahasa asli (id/unknown): 15,197
